In [27]:
import polars as pl
import pandas as pd
import json

In [28]:
# --- Connexion au serveur postgres ---
uri = "postgresql://p8:p8@localhost:5433/meteoforecast"

In [29]:
# --- Fonction de profiling ---

def renseigne(df): 
    n = df.height
    nulls = df.null_count().row(0)
    profil = pl.DataFrame({
        'colonne': df.columns, 
        'type':[str(t)for t in df.dtypes],
        'longueur_max':[df[c].cast(pl.Utf8).str.len_chars().max() for c in df.columns],
        'non_null':[n - x for x in nulls],
        'nan':list(nulls),
        "% nan": [round(x / n * 100, 2) for x in nulls],
        'valeur_unique':[df[c].n_unique()for c in df.columns]
    })
    return profil

#cols_airbnb = renseigne(df) # Instance le df profilé
# cols_airbnb.write_csv('../docs/cols_airbnb.csv') # Produit un csv du profil
#display(cols_airbnb) # Affiche le résultat de la fonctioin de profiling

In [30]:
# --- Liste de toute les tables sur le serveur "raw" ---

tables = pl.read_database_uri(
    "SELECT tablename FROM pg_tables WHERE schemaname = 'raw'", uri
    )["tablename"].to_list()

In [31]:
# --- Extrait les tables pg, applique la fct et produit le fichier xlsx ---

with pd.ExcelWriter("./data/profiling_raw.xlsx") as writer:
    for table in tables:
        df = pl.read_database_uri(f"SELECT * FROM raw.{table}", uri)
        profil = renseigne(df)
        profil.to_pandas().to_excel(writer, sheet_name = table[:31], index=False)

In [36]:
# --- json ---

df_infoclimat = pl.read_database_uri("SELECT * FROM raw.infoclimat", uri)
df_open_meteo = pl.read_database_uri("SELECT * FROM raw.open_meteo_roubaix", uri)

stations = json.loads(df_infoclimat["stations"][0])
df_ic_stations = pl.DataFrame(stations) 
hourly = json.loads(df_infoclimat["hourly"][0])

mesures = []
for station, liste in hourly.items():
    if station == "_params": 
        continue
    mesures.extend(liste) 
df_ic_hourly = pl.DataFrame(mesures, strict=False, infer_schema_length=None) 

open_meteo = json.loads(df_open_meteo["hourly"][0])
df_om = pl.DataFrame(open_meteo, strict=False)                      

with pd.ExcelWriter("./data/profiling_json.xlsx") as writer:   # (4)
    renseigne(df_om).to_pandas().to_excel(writer, sheet_name="om_mesures", index=False)
    renseigne(df_ic_stations).to_pandas().to_excel(writer, sheet_name="ic_stations", index=False)
    renseigne(df_ic_hourly).to_pandas().to_excel(writer, sheet_name="ic_hourly", index=False)